# 3 - Assigning the new market

Prospects are **assigned**, not re-clustered. Re-clustering them would produce segments numbered and shaped differently from the existing market's, so no campaign could carry across. The fitted encoder and prototypes are applied unchanged.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, joblib
from IPython.display import Image, display
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

from src import config as C, data as D, features as F, viz
viz.set_style()

from src import cluster as CL, score
import json

In [ ]:
R = json.load(open(C.RESULTS_JSON, encoding="utf-8"))
art = joblib.load(C.OUTPUTS / "segmentation_model.joblib")
print("k =", art["k"], "| gamma = %.3f" % art["gamma"])
print("prototype for segment 0:"); art["prototypes"][0]

## The deliverable

In [ ]:
scored = pd.read_csv(C.OUTPUTS / "segmented_new_customers.csv")
print(scored.shape)
scored.head()

In [ ]:
score.assignment_summary(scored)

## How confident is each assignment?

`Separation` is how much closer the winning prototype is than the runner-up: 0 means the customer sits midway between two segments, 1 means they sit on the prototype. In a continuum a large borderline group is expected, and saying so is more useful than emitting a confident-looking segment id.

In [ ]:
scored["Assignment_Tier"].value_counts()

## Does the new market have the same shape as the old one?

In [ ]:
pd.DataFrame(R["segment_mix_comparison"])

In [ ]:
display(Image(filename=str(C.FIGURES / "11_assignment.png")))

## Using the saved model on future prospects

The artefact carries the encoder, the prototypes and gamma - everything needed to assign a new customer without re-running the clustering.

In [ ]:
new = D.load_prospects().head(5)
scaled, _ = art["encoder"].transform(new)
labels, conf = CL.assign(scaled, art["prototypes"], art["gamma"])
conf.assign(Tier=score.tier(conf["Separation"]))